In [0]:
catalog = "neobank"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.metadata")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.metadata.tables (
    table_id INT,
    table_name STRING,
    source_system STRING,
    source_path STRING,
    target_layer STRING,
    bronze_schema STRING,
    silver_schema STRING,
    active_flag BOOLEAN,
    load_order INT,
    created_date TIMESTAMP
)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.metadata.table_parameters (
    table_id INT,
    param_name STRING,
    param_value STRING,
    created_date TIMESTAMP
)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.metadata.table_watermarks (
    table_id INT,
    last_watermark TIMESTAMP,
    last_run_id STRING
)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.metadata.pipeline_runs (
    run_id STRING,
    table_id INT,
    layer STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    records_processed BIGINT,
    error_message STRING 
)
""")

In [0]:
base = "/Volumes/neobank/neobank/neobank/raw"

spark.sql(f"""
INSERT INTO {catalog}.metadata.tables Values
(1, 'customers', 'blob', '{base}/customers', 'silver', 'bronze', 'silver', true, 1, current_timestamp()),
(2, 'accounts', 'blob', '{base}/accounts', 'silver', 'bronze', 'silver', true, 2, current_timestamp()),
(3, 'transactions', 'blob', '{base}/transactions', 'silver', 'bronze', 'silver', true, 3, current_timestamp()),
(4, 'branches', 'blob', '{base}/branches', 'silver', 'bronze', 'silver', true, 4, current_timestamp()),
(5, 'credit_bureau_reports', 'blob', '{base}/credit_bureau_reports', 'silver', 'bronze', 'silver', true, 5, current_timestamp()),
(6, 'payment_gateway_logs', 'blob', '{base}/payment_gateway_logs', 'silver', 'bronze', 'silver', true, 6, current_timestamp()),
(7, 'branch_performance', 'silver', NULL, 'gold', NULL, NULL, true, 1, current_timestamp()),
(8,  'customer_360',              'silver', NULL, 'gold', NULL, NULL, true, 2, current_timestamp()),
(9,  'risk_customer_summary',     'silver', NULL, 'gold', NULL, NULL, true, 3, current_timestamp()),
(10, 'daily_bank_kpi',            'silver', NULL, 'gold', NULL, NULL, true, 4, current_timestamp()),
(11, 'transaction_channel_summary','silver', NULL, 'gold', NULL, NULL, true, 5, current_timestamp())
""")

In [0]:
spark.sql(f"""
INSERT INTO {catalog}.metadata.table_parameters VALUES
 (1, 'loadtype', 'merge', current_timestamp()),
 (1, 'primary_key', 'customer_id', current_timestamp()),
 (1, 'watermark_column', 'insert_timestamp', current_timestamp()),

 (2, 'loadtype', 'merge', current_timestamp()),
 (2, 'primary_key', 'account_id', current_timestamp()),
 (2, 'watermark_column', 'insert_timestamp', current_timestamp()),

 (3, 'loadtype', 'append', current_timestamp()),
 (3, 'watermark_column', 'insert_timestamp', current_timestamp()),

 (4, 'loadtype', 'full', current_timestamp()),

 (5, 'loadtype', 'merge', current_timestamp()),
 (5, 'primary_key', 'customer_id, bureau_pull_date', current_timestamp()),
 (5, 'watermark_column', 'insert_timestamp', current_timestamp()),

 (6, 'loadtype', 'append', current_timestamp()),
 (6, 'watermark_column', 'insert_timestamp', current_timestamp())
""")

spark.sql(f"""
INSERT INTO {catalog}.metadata.table_watermarks VALUES
 (1, TIMESTAMP'1900-01-01 00:00:00', NULL),
 (2, TIMESTAMP'1900-01-01 00:00:00', NULL),
 (3, TIMESTAMP'1900-01-01 00:00:00', NULL),
 (5, TIMESTAMP'1900-01-01 00:00:00', NULL),
 (6, TIMESTAMP'1900-01-01 00:00:00', NULL)
""")

In [0]:
for t in ["tables", "table_parameters", "table_watermarks", "pipeline_runs"]:
    print(t, spark.table(f"{catalog}.metadata.{t}").count())

spark.sql(f"""
SELECT t.table_id, t.table_name, t.source_system, t.target_layer, 
       MAX(CASE WHEN p.param_name = 'loadtype' THEN p.param_value END) AS load_type,
       MAX(CASE WHEN p.param_name = 'primary_key' THEN p.param_value END) AS primary_key,
       MAX(CASE WHEN p.param_name = 'watermark_column' THEN p.param_value END) AS watermark_column
FROM {catalog}.metadata.tables t
LEFT JOIN {catalog}.metadata.table_parameters p ON t.table_id = p.table_id
GROUP BY t.table_id, t.table_name, t.source_system, t.target_layer
ORDER BY t.table_id
""").show(truncate = False)

for r in spark.table(f"{catalog}.metadata.tables").filter("source_system = 'blob'").collect():
    print(r.table_name, [f.name for f in dbutils.fs.ls(r.source_path)])

In [0]:
print("--- parameters: duplicates (count should be 1 each) ---")
spark.sql(f"""
SELECT table_id, param_name, param_value, COUNT(*) AS copies
FROM {catalog}.metadata.table_parameters
GROUP BY table_id, param_name, param_value
HAVING COUNT(*) >= 1
ORDER BY table_id
""").show(50, truncate=False)

print("--- watermarks: all rows ---")
spark.sql(f"SELECT * FROM {catalog}.metadata.table_watermarks ORDER BY table_id").show(truncate=False)